In [7]:
# import statements
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import isnull, when, count, col
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

sc = SparkContext.getOrCreate()
spark = SparkSession(sc) 

In [9]:
print(spark.version)

3.5.5


In [21]:
import os
print(os.path.abspath('Mumbai.csv'))


C:\Users\USER\Mumbai.csv


In [25]:
# Read  the data from  dbfs:/databricks-datasets/Mumbai/Mumbai.data  into Saprk dataframe  
df=spark.read.format("csv").option("header", "false").option("inferschema","True").load('C:\\Users\\USER\\Mumbai.csv')

In [29]:
# Read the data from the file into a Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .load("C:\\Users\\USER\\Mumbai.csv")
df.show()


+--------+----+------------------+---------------+------+----------------+---------+------------+-----------------+------------+-------------------+-----------+------------+--------+--------------+----+---------+------+------------+-----------+----------+------------+---------+----------------+--------+--------------+-------------+----+----+------------------+-------------+----+---------------+---------+----------+----+-----------+----+--------+------------+
|     _c0| _c1|               _c2|            _c3|   _c4|             _c5|      _c6|         _c7|              _c8|         _c9|               _c10|       _c11|        _c12|    _c13|          _c14|_c15|     _c16|  _c17|        _c18|       _c19|      _c20|        _c21|     _c22|            _c23|    _c24|          _c25|         _c26|_c27|_c28|              _c29|         _c30|_c31|           _c32|     _c33|      _c34|_c35|       _c36|_c37|    _c38|        _c39|
+--------+----+------------------+---------------+------+----------------+

In [31]:
column_names = [
    "Price", "Area", "Location", "No. of Bedrooms", "Resale", "MaintenanceStaff", 
    "Gymnasium", "SwimmingPool", "LandscapedGardens", "JoggingTrack", "RainWaterHarvesting", 
    "IndoorGames", "ShoppingMall", "Intercom", "SportsFacility", "ATM", "ClubHouse", 
    "School", "24X7Security", "PowerBackup", "CarParking", "StaffQuarter", "Cafeteria", 
    "MultipurposeRoom", "Hospital", "WashingMachine", "Gasconnection", "AC", "Wifi", 
    "Children_splayarea", "LiftAvailable", "BED", "VaastuCompliant", "Microwave", "GolfCourse", 
    "TV", "DiningTable", "Sofa", "Wardrobe", "Refrigerator"
]

for new_col, old_col in zip(column_names, df.columns):
    df = df.withColumnRenamed(old_col, new_col)

In [57]:
from pyspark.sql.functions import isnull, when, count, col

#df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show()
#df.select([count(when(isnull(col(c)), c)).alias(c) for c in df.columns]).show()
#df.select([sum(when(isnull(col(c)), 1).otherwise(0)).alias(c) for c in df.columns]).show()
# Step 1: Use agg() to count null values for each column
# Step 1: Aggregate null values for each column
# Use agg() to count null values for each column


In [53]:
from pyspark.sql.functions import isnull, when, count, col

# Step 1: Escape the column names properly
def safe_col(column_name):
    return col(f"`{column_name}`")

# Step 2: Aggregate null counts for each column
null_counts = df.agg(*[
    count(when(isnull(safe_col(c)), 1)).alias(c) 
    for c in df.columns
])

# Step 3: Collect the result
null_counts_row = null_counts.collect()[0].asDict()

# Step 4: Find columns which have null values
columns_with_nulls = [col_name for col_name, null_count in null_counts_row.items() if null_count > 0]

# Step 5: Print the columns with nulls
print("Columns with NULL values:")
for col_name in columns_with_nulls:
    print(f"{col_name} ({null_counts_row[col_name]} nulls)")


Columns with NULL values:


In [55]:
df.printSchema()

root
 |-- Price: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- No. of Bedrooms: string (nullable = true)
 |-- Resale: string (nullable = true)
 |-- MaintenanceStaff: string (nullable = true)
 |-- Gymnasium: string (nullable = true)
 |-- SwimmingPool: string (nullable = true)
 |-- LandscapedGardens: string (nullable = true)
 |-- JoggingTrack: string (nullable = true)
 |-- RainWaterHarvesting: string (nullable = true)
 |-- IndoorGames: string (nullable = true)
 |-- ShoppingMall: string (nullable = true)
 |-- Intercom: string (nullable = true)
 |-- SportsFacility: string (nullable = true)
 |-- ATM: string (nullable = true)
 |-- ClubHouse: string (nullable = true)
 |-- School: string (nullable = true)
 |-- 24X7Security: string (nullable = true)
 |-- PowerBackup: string (nullable = true)
 |-- CarParking: string (nullable = true)
 |-- StaffQuarter: string (nullable = true)
 |-- Cafeteria: string (nullable = true)
 |-- MultipurposeRo

In [63]:
# Create SparkSession
spark = SparkSession.builder.getOrCreate()
# Create SparkSession
spark = SparkSession.builder.getOrCreate()

# Step 1: Rename all columns (replace spaces and dots with underscore)
for old_name in df.columns:
    new_name = old_name.replace(" ", "_").replace(".", "_")
    df = df.withColumnRenamed(old_name, new_name)

# Step 2: List all string columns
input_cols = [field.name for field in df.schema.fields if field.dataType.simpleString() == 'string']

# Step 3: Create output column names (add _indexed to each)
output_cols = [col + "_indexed" if col != 'Price' else 'label' for col in input_cols]

# Step 4: Create one multi-column StringIndexer
indexer = StringIndexer(
    inputCols=input_cols,
    outputCols=output_cols,
    handleInvalid='keep'  # optional, but recommended
).fit(df)

# Step 5: Transform the dataframe
df_indexed = indexer.transform(df)

# Step 6: Drop the original string columns
df_final = df_indexed.drop(*input_cols)

# Step 7: Show final dataframe
df_final.show(5, False)
df_final.printSchema()

+-----+------------+----------------+-----------------------+--------------+------------------------+-----------------+--------------------+-------------------------+--------------------+---------------------------+-------------------+--------------------+----------------+----------------------+-----------+-----------------+--------------+--------------------+-------------------+------------------+--------------------+-----------------+------------------------+----------------+----------------------+---------------------+----------+------------+--------------------------+---------------------+-----------+-----------------------+-----------------+------------------+----------+-------------------+------------+----------------+--------------------+
|label|Area_indexed|Location_indexed|No__of_Bedrooms_indexed|Resale_indexed|MaintenanceStaff_indexed|Gymnasium_indexed|SwimmingPool_indexed|LandscapedGardens_indexed|JoggingTrack_indexed|RainWaterHarvesting_indexed|IndoorGames_indexed|ShoppingM

In [65]:
# Step 1: Rename columns (replace spaces and dots with underscore)
for old_name in df.columns:
    new_name = old_name.replace(" ", "_").replace(".", "_")
    df = df.withColumnRenamed(old_name, new_name)

# Step 2: List all string columns
input_cols = [field.name for field in df.schema.fields if field.dataType.simpleString() == 'string']

# Step 3: Create output columns (add _indexed to each string col)
output_cols = [col + "_indexed" if col != 'Price' else 'label' for col in input_cols]

# Step 4: StringIndexer for all string columns
indexer = StringIndexer(
    inputCols=input_cols,
    outputCols=output_cols,
    handleInvalid='keep'
).fit(df)

# Step 5: Transform the dataframe
df_indexed = indexer.transform(df)

# Step 6: Drop original string columns
df_clean = df_indexed.drop(*input_cols)

# Step 7: Now create features columns (all columns except 'label')
feature_cols = [col for col in df_clean.columns if col != 'label']

# Step 8: Assemble features into one 'features' column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_final = assembler.transform(df_clean)

# Step 9: Final output
df_final.select("features", "label").show(5, False)
df_final.printSchema()

+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                          |label|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|[1140.0,310.0,7.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0]|977.0|
|[60.0,0.0,1.0,0.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]    |217.0|
|[2.0,0.0,1.0,0.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0

In [69]:
# Split the assembled DataFrame into train and test
df_train, df_test = df_final.randomSplit([0.67, 0.33], seed=2020)

# (Optional) Check how many rows are in each
print("Training Data Count: ", df_train.count())
print("Test Data Count: ", df_test.count())


Training Data Count:  5236
Test Data Count:  2484


In [71]:
# Build the LogisticRegression object 'lr' by setting the required parameters
lr = LogisticRegression(featuresCol="features", labelCol="label")

# fit the LogisticRegression object on the training data
lrmodel = lr.fit(df_train)

In [73]:
#This LogisticRegressionModel can be used as a transformer to perform prediction on the testing data
predictonDF = lrmodel.transform(df_test)
evaluator = BinaryClassificationEvaluator()

# Calculate the accracy and print its value
accuracy = predictonDF.filter(predictonDF.label == predictonDF.prediction).count()/float(predictonDF.count())
print("Accuracy = ", accuracy)

Accuracy =  0.026570048309178744
